# Validación y métrica del negocio de mi proyecto

Evalúo el modelo final de la Semana 11 (bosque con `min_samples_leaf=20`) y el segundo mejor (la red neuronal) con validación cruzada de dos formas: temporal, que es la que corresponde a mis datos, y al azar, para compararlas. Después calculo una métrica que refleja el costo real de equivocarse.

La validación cruzada se hace solo con el conjunto de entrenamiento (antes del 30 de abril de 2015). La prueba no se toca hasta el final.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error

from pipeline_proyecto import RUTA_DATOS, cargar_y_dividir, construir_preparacion

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()

# para la validacion temporal hay que ordenar el entrenamiento por fecha
fechas = pd.read_csv(RUTA_DATOS, usecols=["DATE_KEY"])["DATE_KEY"]
orden = fechas.loc[X_ent.index].sort_values(kind="stable").index
X_ent, y_ent = X_ent.loc[orden], y_ent.loc[orden]
print("Entrenamiento ordenado por fecha:", X_ent.shape)

Entrenamiento ordenado por fecha: (159871, 11)


## La métrica del negocio

La predicción sirve para decidir cuánto pedir de cada producto. Los dos errores no cuestan lo mismo:

- **Subestimar** (pedir de menos): el producto se acaba y se pierde la venta, con su margen y a veces al cliente. Es el error más caro.
- **Sobreestimar** (pedir de más): queda inventario parado, que cuesta almacenarlo y, si es perecedero, tirarlo.

No tengo los costos reales, así que uso una aproximación: cada unidad que falta cuesta unas 3 veces lo que cuesta una unidad que sobra. El costo del negocio por fila es `3 × unidades que faltaron + 1 × unidades que sobraron`, en unidades de inventario de sobra. Lo que importa es la comparación entre modelos, no el valor absoluto.

In [2]:
COSTO_FALTANTE = 3
COSTO_SOBRANTE = 1


def costo_negocio(real, estimado):
    faltante = np.maximum(np.asarray(real) - estimado, 0)
    sobrante = np.maximum(estimado - np.asarray(real), 0)
    return float((COSTO_FALTANTE * faltante + COSTO_SOBRANTE * sobrante).mean())


def modelo_final():
    return RandomForestRegressor(n_estimators=100, min_samples_leaf=20, random_state=42, n_jobs=-1)


def modelo_segundo():
    return MLPRegressor(hidden_layer_sizes=(32, 16), early_stopping=True, max_iter=200, random_state=42)

## Validación cruzada: temporal y al azar

In [ ]:
esquemas = {
    "temporal": TimeSeriesSplit(n_splits=5),
    "al azar": KFold(n_splits=5, shuffle=True, random_state=42),
}
candidatos = {
    "bosque min_samples_leaf=20": modelo_final,
    "red neuronal (32, 16)": modelo_segundo,
}

filas = []
for nombre_esquema, esquema in esquemas.items():
    for nombre_modelo, fabrica in candidatos.items():
        for numero, (ent, val) in enumerate(esquema.split(X_ent), start=1):
            pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", fabrica())])
            pipeline.fit(X_ent.iloc[ent], y_ent.iloc[ent])
            estimado = pipeline.predict(X_ent.iloc[val])
            real = y_ent.iloc[val]
            filas.append({
                "modelo": nombre_modelo, "esquema": nombre_esquema, "pliegue": str(numero),
                "rmse": root_mean_squared_error(real, estimado),
                "costo_negocio": costo_negocio(real, estimado),
            })
        print("listo:", nombre_esquema, "|", nombre_modelo)

pliegues = pd.DataFrame(filas)

In [ ]:
# promedio y desviacion de cada modelo y esquema, y todo junto en un solo archivo
resumen = pliegues.groupby(["modelo", "esquema"])[["rmse", "costo_negocio"]].agg(["mean", "std"])
resumen.columns = ["_".join(c) for c in resumen.columns]
resumen = resumen.round(4)
print(resumen)

promedios = pliegues.groupby(["modelo", "esquema"])[["rmse", "costo_negocio"]].mean().reset_index().assign(pliegue="promedio")
desviaciones = pliegues.groupby(["modelo", "esquema"])[["rmse", "costo_negocio"]].std().reset_index().assign(pliegue="desviacion")
tabla = pd.concat([pliegues, promedios, desviaciones], ignore_index=True).round(4)
tabla.to_csv("validacion_proyecto.csv", index=False)
tabla[tabla["pliegue"].isin(["promedio", "desviacion"])]

                                     rmse_mean  rmse_std  costo_negocio_mean  \
modelo                     esquema                                             
bosque min_samples_leaf=20 al azar      2.9957    0.3873              2.0448   
                           temporal     3.2488    0.3899              2.2891   
red neuronal (32, 16)      al azar      2.9643    0.3857              2.0797   
                           temporal     3.2350    0.4045              2.3561   

                                     costo_negocio_std  
modelo                     esquema                      
bosque min_samples_leaf=20 al azar              0.0551  
                           temporal             0.1334  
red neuronal (32, 16)      al azar              0.0778  
                           temporal             0.0983  


,modelo,esquema,pliegue,rmse,costo_negocio
20,bosque min_samples_leaf=20,al azar,promedio,2.9957,2.0448
21,bosque min_samples_leaf=20,temporal,promedio,3.2488,2.2891
22,"red neuronal (32, 16)",al azar,promedio,2.9643,2.0797
23,"red neuronal (32, 16)",temporal,promedio,3.2350,2.3561
24,bosque min_samples_leaf=20,al azar,desviacion,0.3873,0.0551
25,bosque min_samples_leaf=20,temporal,desviacion,0.3899,0.1334
26,"red neuronal (32, 16)",al azar,desviacion,0.3857,0.0778
27,"red neuronal (32, 16)",temporal,desviacion,0.4045,0.0983


In [ ]:
# la diferencia entre los dos modelos, pliegue por pliegue
for esquema in esquemas:
    a = pliegues[(pliegues.esquema == esquema) & (pliegues.modelo == "bosque min_samples_leaf=20")].set_index("pliegue")["rmse"]
    b = pliegues[(pliegues.esquema == esquema) & (pliegues.modelo == "red neuronal (32, 16)")].set_index("pliegue")["rmse"]
    dif = (b - a)
    print(f"{esquema:9s} red - bosque, por pliegue: {dif.round(3).tolist()} | promedio {dif.mean():.4f}")

temporal  red - bosque, por pliegue: [-0.002, 0.014, -0.009, -0.047, -0.025] | promedio -0.0138
al azar   red - bosque, por pliegue: [-0.046, -0.04, -0.029, -0.017, -0.024] | promedio -0.0314


## La métrica del negocio sobre el conjunto de prueba

In [ ]:
modelos_prueba = {
    "linea base (mediana)": DummyRegressor(strategy="median"),
    "bosque min_samples_leaf=20": modelo_final(),
    "red neuronal (32, 16)": modelo_segundo(),
}
filas = []
for nombre, modelo in modelos_prueba.items():
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)]).fit(X_ent, y_ent)
    estimado = pipeline.predict(X_prueba)
    faltante = np.maximum(y_prueba.to_numpy() - estimado, 0)
    sobrante = np.maximum(estimado - y_prueba.to_numpy(), 0)
    filas.append({
        "modelo": nombre,
        "rmse": round(root_mean_squared_error(y_prueba, estimado), 4),
        "costo_negocio": round(costo_negocio(y_prueba, estimado), 4),
        "unidades_faltantes": round(faltante.sum()),
        "unidades_sobrantes": round(sobrante.sum()),
    })
pd.DataFrame(filas)

,modelo,rmse,costo_negocio,unidades_faltantes,unidades_sobrantes
0,linea base (mediana),3.8348,3.9459,52562,0
1,bosque min_samples_leaf=20,2.9502,2.5982,26189,25263
2,"red neuronal (32, 16)",2.9313,2.6476,27521,23240


Los errores de prueba de esta tabla no coinciden del todo con los de `experimentos_proyecto.ipynb` (por ejemplo, el bosque da 2.9502 aquí y 2.9332 allá, y la red 2.9313 contra 2.9572). La razón es que aquí ordené el entrenamiento por fecha para la validación temporal, y el orden de las filas cambia los remuestreos del bosque, la validación interna del codificador del producto y el orden de los lotes de la red. Es una diferencia de unos 0.02, del mismo tamaño que la diferencia entre el bosque y la red, y por eso no uso el error de prueba para decidir entre los dos.